<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.1-agent-fundamentals/notebooks/exercises-9_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.1: Agent Fundamentals in Code

*Module 9 · AI Agents & Autonomous Systems*

> An LLM answers questions. An agent takes actions. The difference is one loop: observe → think → act → observe the result → think again. This lesson builds that loop from scratch in Python — no frameworks, no magic. You’ll see exactly how the agent decides which tool to call, parses the output, handles errors, and knows when to stop. Every agent framework (LangGraph, CrewAI, AutoGen) is just a fancy wrapper around this loop.

`Agent Loop` · `Observe-Think-Act` · `Tool Selection` · `From Scratch` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — The Minimal Agent — 40 Lines, Zero Frameworks — `01_minimal_agent.py`
2. Step 2 — Structured Agent — Proper Parsing + Error Recovery — `02_structured_agent.py`
3. Step 3 — Gemini Native Agent — Automatic Function Calling — `03_gemini_agent.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · The Minimal Agent — 40 Lines, Zero Frameworks

Observe-think-act in pure Python. See every decision the agent makes.

**`01_minimal_agent.py`** · _Block 1 of 3_

MINIMAL AGENT — The core loop in 40 lines


In [ ]:
# MINIMAL AGENT — The core loop in 40 lines
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# ── Tools the agent can use ──
def search(query: str) -> dict:
    """Search Netsetos knowledge base."""
    db = {"genai":"GenAI: 14999 INR, 146hrs", "refund":"Full 7 days, 50% 8-30, none after 30",
          "python":"Python: 9999 INR, 80hrs", "schedule":"Live 7PM IST daily"}
    return {"result": db.get(query.lower().split()[0], "Not found")}

def calculate(expression: str) -> dict:
    """Calculate a math expression."""
    try: return {"result": eval(expression, {"__builtins__":{}})}
    except: return {"error": "Invalid expression"}

TOOLS = {"search": search, "calculate": calculate}

# ── The Agent Loop ──
def agent(task, max_steps=5):
    model = genai.GenerativeModel("gemini-2.0-flash")
    history = []

    for step in range(max_steps):
        # THINK: ask LLM what to do next
        prompt = (f"Task: {task}\n\nHistory:\n" +
                  "\n".join(history) +
                  f"\n\nAvailable tools: search(query), calculate(expression)\n"
                  f"Respond with EXACTLY one of:\n"
                  f'  TOOL: tool_name | argument\n'
                  f'  ANSWER: your final answer\n')

        response = model.generate_content(prompt).text.strip()
        print(f"  Step {step+1}: {response[:80]}")

        # DONE: agent has an answer
        if response.startswith("ANSWER:"):
            return response[7:].strip()

        # ACT: parse and execute tool call
        if response.startswith("TOOL:"):
            parts = response[5:].strip().split("|", 1)
            tool_name = parts[0].strip()
            tool_arg = parts[1].strip() if len(parts) > 1 else ""

            if tool_name in TOOLS:
                result = TOOLS[tool_name](tool_arg)  # EXECUTE
                # OBSERVE: add result to history
                history.append(f"Used {tool_name}({tool_arg}) → {json.dumps(result)}")
            else:
                history.append(f"Error: unknown tool '{tool_name}'")

    return "Max steps reached"

# ── Run ──
print("Agent (from scratch):\n")
for task in ["How much does the GenAI course cost?",
            "What is 14999 with 18% GST?",
            "What is the refund policy and when are live classes?"]:
    print(f"Task: {task}")
    answer = agent(task)
    print(f"Answer: {answer}\n")


> **What just happened?** “What is 14999 with 18% GST?” → Step 1: TOOL: calculate | 14999*1.18 → Result: 17698.82 → Step 2: ANSWER: 17,698.82 INR. The agent DECIDED to use the calculator, EXECUTED it, OBSERVED the result, then ANSWERED. For “refund policy and live classes”, it called search TWICE (refund, then schedule), then combined both into one answer. This is the entire agent pattern: loop of think → act → observe until done. Every framework wraps this.


### Step 2 · Structured Agent — Proper Parsing + Error Recovery

Production-quality agent with JSON tool calls, retry logic, and step logging.

**`02_structured_agent.py`** · _Block 2 of 3_

STRUCTURED AGENT — JSON parsing + error recovery


In [ ]:
# STRUCTURED AGENT — JSON parsing + error recovery
import google.generativeai as genai, json, os, time
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class Agent:
    def __init__(self, tools, system="", max_steps=8):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.tools = {t.__name__: t for t in tools}
        self.system = system or "You are a helpful assistant with tools."
        self.max_steps = max_steps
        self.trace = []

    def _build_prompt(self, task, history):
        tool_desc = "\n".join(f"- {n}: {t.__doc__}" for n,t in self.tools.items())
        return (f"{self.system}\n\nTools:\n{tool_desc}\n\n"
                f"Task: {task}\n\nHistory:\n" + "\n".join(history) +
                f'\n\nRespond with JSON:\n'
                f'{{"action":"tool","name":"...","args":{{"...":"..."}}}}\n'
                f'OR {{"action":"answer","text":"your final answer"}}\n'
                f'Return ONLY valid JSON.')

    def _parse(self, text):
        text = text.strip()
        if text.startswith("```"): text = text.split("\n",1)[1].rsplit("```",1)[0]
        try: return json.loads(text)
        except: return {"action":"answer", "text":text}

    def run(self, task):
        history, self.trace = [], []
        for step in range(self.max_steps):
            start = time.time()
            prompt = self._build_prompt(task, history)
            raw = self.model.generate_content(prompt).text
            parsed = self._parse(raw)
            lat = (time.time()-start)*1000

            if parsed.get("action") == "answer":
                self.trace.append({"step":step+1, "type":"answer", "ms":lat})
                return {"answer":parsed["text"], "steps":step+1, "trace":self.trace}

            if parsed.get("action") == "tool":
                name = parsed.get("name","")
                args = parsed.get("args",{})
                if name in self.tools:
                    try:
                        result = self.tools[name](**args)
                        history.append(f"[{name}({args})] → {json.dumps(result)}")
                        self.trace.append({"step":step+1,"tool":name,"args":args,"ok":True,"ms":lat})
                    except Exception as e:
                        history.append(f"[{name}] ERROR: {e}")
                        self.trace.append({"step":step+1,"tool":name,"ok":False,"error":str(e)})
                else:
                    history.append(f"Unknown tool: {name}")
        return {"answer":"Max steps", "steps":self.max_steps, "trace":self.trace}

# ── Tools ──
def search(query=""):
    """Search Netsetos knowledge base for course info, policies, schedules."""
    db = {"genai":{"course":"GenAI","price":14999,"hours":146},
          "refund":{"7days":"100%","30days":"50%","after":"0%"}}
    return db.get(query.lower().split()[0], {"info":"Not found"})

def calculate(expression=""):
    """Evaluate a math expression like '14999 * 1.18'."""
    return {"result": eval(expression, {"__builtins__":{}})}

def get_time():
    """Get current date and time."""
    from datetime import datetime
    return {"now": datetime.now().strftime("%Y-%m-%d %H:%M")}

agent = Agent([search, calculate, get_time],
              system="You are a Netsetos support agent. Answer from tools.")

print("Structured Agent:\n")
for task in ["What is the GenAI course price with GST (18%)?",
            "What is the refund policy?",
            "What time is it right now?"]:
    r = agent.run(task)
    print(f"  Task: {task}")
    print(f"  Answer: {r['answer'][:100]}")
    print(f"  Steps: {r['steps']}, Trace: {[t.get('tool','answer') for t in r['trace']]}\n")


### Step 3 · Gemini Native Agent — Automatic Function Calling

Gemini handles the tool loop internally. Compare: manual vs automatic.

**`03_gemini_agent.py`** · _Block 3 of 3_

GEMINI NATIVE AGENT — Auto function calling


In [ ]:
# GEMINI NATIVE AGENT — Auto function calling
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def search_courses(topic: str) -> dict:
    """Search Netsetos course catalog by topic."""
    courses = {"genai":{"name":"GenAI","price":14999}, "python":{"name":"Python","price":9999}}
    return courses.get(topic.lower(), {"error":"Not found"})

def calculate_price(amount: float, tax_pct: float = 18.0) -> dict:
    """Calculate price with tax."""
    total = amount * (1 + tax_pct/100)
    return {"subtotal":amount, "tax":round(amount*tax_pct/100,2), "total":round(total,2)}

def check_refund(days_since_purchase: int) -> dict:
    """Check refund eligibility based on days since purchase."""
    if days_since_purchase <= 7: return {"refund":"100%","eligible":True}
    if days_since_purchase <= 30: return {"refund":"50%","eligible":True}
    return {"refund":"0%","eligible":False}

# Agent with auto function calling
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    tools=[search_courses, calculate_price, check_refund],
    system_instruction="You are a Netsetos support agent. Use tools to answer questions."
)
chat = model.start_chat(enable_automatic_function_calling=True)

print("Gemini Native Agent:\n")
for q in ["How much is the GenAI course with tax?",
          "I bought the course 10 days ago, can I get a refund?",
          "Compare GenAI and Python course prices with GST"]:
    resp = chat.send_message(q)
    print(f"  Q: {q}")
    print(f"  A: {resp.text.strip()[:120]}\n")

print("  Manual agent (01, 02): you control the loop, parse, retry")
print("  Gemini agent (03): one flag, auto loop. Less control, faster dev.")
print("  Know BOTH: manual for learning + custom. Auto for production speed.")


> **What just happened?** Multi-step reasoning is where agents shine over simple function calling. The agent dynamically decides what to do next based on previous results. If the weather API fails, it retries or uses an alternative. If the calculation needs more data, it fetches it. This adaptive behavior is the defining characteristic of agentic systems.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
